# feature name sources -- v1

**Kernel: `fttl-v1` (env-v1, Python 3.5.6).** The v2/v3 twin of this notebook is
`feature_name_sources_v2v3.ipynb`; this one is separate because env-v1 cannot run it:
no f-strings, and `src/config.py` cannot even be imported here (it opens with
`from __future__ import annotations`, 3.7+). Paths come from `shap_kit_v1`, v1's stand-in
for config.

v1 is the version where this question actually bites. xgboost 0.72 often records only
positional stand-ins (`f0`, `f1`, ...) instead of names, and the code paths disagree about
what to do with them:

| code | rejects `f0`/`f1`/... ? | how many attributes it searches |
|---|---|---|
| `src/shap_kit_v1.py` `model_feature_names` | **yes** -> returns `[]` -> registry fallback | 3 |
| `features/extract_features_v1.py` | no -> writes them into the registry | 1 + 11 |

So the registry can end up holding either real names or `f0..fN`, and which one decides
whether `00_SHAP_v1.ipynb`'s fallback works at all. Nothing is written to disk here.


In [ ]:
import json
import os
import re
import sys

import pandas as pd

ROOT = os.getcwd()
while not os.path.exists(os.path.join(ROOT, 'src', 'config.py')) and ROOT != os.path.dirname(ROOT):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, os.path.join(ROOT, 'src'))

import shap_kit_v1 as sk

print('kernel  : {}'.format(sys.executable))
print('version : {}  (xgboost pin {})'.format(sk.VERSION, sk.XGBOOST_PIN))
print('registry: {}'.format(sk.REGISTRY_PATH))


## 1 · Load the estimator

Same two knobs as `00_SHAP_v1.ipynb`: name the repo folder, or give the pickle path outright.


In [ ]:
V1_REPO_DIR = ''
MODEL_PATH = ''
assert V1_REPO_DIR or MODEL_PATH, 'set V1_REPO_DIR (folder inside model_repos/real) or MODEL_PATH'
if not MODEL_PATH:
    MODEL_PATH = os.path.join(ROOT, 'model_repos', 'real', V1_REPO_DIR,
                              'outputs', 'fasttracker_xgb.pkl')
print('model: {}'.format(MODEL_PATH))

est = sk.load_estimator(MODEL_PATH)   # unwraps a Pipeline to its final step, if it is one
print('  {}'.format(type(est).__name__))


## 2 · Ask every source separately

The first three are what `shap_kit_v1.model_feature_names` walks. The rest are the extra
attributes `extract_features_v1.py` searches (`COLUMN_ATTRS`) -- the reason the registry can
answer when the pickle path comes up empty.


In [ ]:
PLACEHOLDER = re.compile(r'^f\d+$')

def probe(label, getter):
    try:
        names = getter()
    except Exception as exc:
        return {'source': label, 'available': False, 'n': 0, 'placeholder': None,
                'first_4': '', 'note': '{}: {}'.format(type(exc).__name__, exc)}, None
    if names is None:
        return {'source': label, 'available': False, 'n': 0, 'placeholder': None,
                'first_4': '', 'note': 'attribute is None'}, None
    try:
        names = [str(n) for n in names]
    except TypeError:
        return {'source': label, 'available': False, 'n': 0, 'placeholder': None,
                'first_4': '', 'note': 'not list-like'}, None
    if not names:
        return {'source': label, 'available': False, 'n': 0, 'placeholder': None,
                'first_4': '', 'note': 'empty list'}, None
    return {'source': label, 'available': True, 'n': len(names),
            'placeholder': all(PLACEHOLDER.match(n) for n in names),
            'first_4': ', '.join(names[:4]), 'note': ''}, names

def attr_getter(obj, attr):
    return lambda: getattr(obj, attr, None)

PROBES = [('booster.feature_names', lambda: est.get_booster().feature_names),
          ('estimator.feature_name_', attr_getter(est, 'feature_name_'))]
for attr in sk.__dict__.get('COLUMN_ATTRS', ()) or (
        'feature_names_in_', 'columns', 'columns_', 'feature_names', 'feature_names_',
        'input_columns', 'input_columns_', 'cols', 'cols_', 'variables', 'variables_'):
    PROBES.append(('estimator.' + attr, attr_getter(est, attr)))

NAMES = {}
rows = []
for label, getter in PROBES:
    row, names = probe(label, getter)
    rows.append(row)
    if names is not None:
        NAMES[label] = names

display(pd.DataFrame(rows).set_index('source'))
print('sources that answered: {}'.format(list(NAMES)))


## 3 · What the two code paths would each conclude

This is the whole point of the notebook: the same pickle, read by the two ladders that
actually run in this repo, and what each of them ends up with.


In [ ]:
kit = sk.model_feature_names(est)
print('shap_kit_v1.model_feature_names()  -> {} name(s){}'.format(
    len(kit), '' if kit else '   <- EMPTY, so 00_SHAP_v1 falls back to the registry'))
if kit:
    print('   first 4: {}'.format(kit[:4]))

raw = NAMES.get('booster.feature_names') or []
if raw and all(PLACEHOLDER.match(n) for n in raw):
    print('\nthe booster DOES expose names, but they are positional stand-ins {}...'.format(raw[:3]))
    print('shap_kit_v1 rejects those on purpose; extract_features_v1.py does NOT, so the')
    print('registry would be written with f0..fN -- which then cannot index a real matrix.')
elif raw:
    print('\nthe booster exposes REAL names, so neither the registry fallback nor the')
    print('placeholder question applies to this pickle.')


## 4 · Do the sources agree with each other, and with the registry?


In [ ]:
labels = list(NAMES)
if len(labels) < 2:
    print('only {} source answered -- nothing to cross-check.'.format(len(labels)))
else:
    rows = []
    for i in range(len(labels)):
        for j in range(i + 1, len(labels)):
            a, b = NAMES[labels[i]], NAMES[labels[j]]
            same_set = sorted(a) == sorted(b)
            first_bad = None
            if same_set:
                for k in range(min(len(a), len(b))):
                    if a[k] != b[k]:
                        first_bad = k
                        break
            rows.append({'A': labels[i], 'B': labels[j], 'same set': same_set,
                         'same ORDER': a == b, 'first mismatch at': first_bad})
    display(pd.DataFrame(rows))

if not os.path.exists(sk.REGISTRY_PATH):
    print('\nno registry at {} -- build it with features/extract_features_v1.py'.format(
        sk.REGISTRY_PATH))
else:
    with open(sk.REGISTRY_PATH, encoding='utf-8') as fh:
        reg = json.load(fh)
    reg_names = [str(c) for c in (reg.get('model_features') or [])]
    print('\nregistry model_features        : {}'.format(len(reg_names)))
    print('registry model_features_source : {}'.format(
        reg.get('model_features_source',
                '<< field absent -- registry predates it; re-run extract_features_v1.py >>')))
    for label in NAMES:
        print('  vs {:<38} set={:<6} order={}'.format(
            label, str(sorted(NAMES[label]) == sorted(reg_names)),
            NAMES[label] == reg_names))


## 5 · What the phi on disk were actually computed against

Each attribution meta's `feature_names` is the column order the saved φ belong to. If it
disagrees with whichever source is authoritative here, those φ are attached to the wrong
feature names.


In [ ]:
shap_dir = os.path.dirname(sk.attributions_csv_path(sk.SPLITS[0]))
metas = sorted(f for f in os.listdir(shap_dir)
               if f.startswith('v1_attributions_') and f.endswith('_meta.json')) \
        if os.path.isdir(shap_dir) else []
print('attribution metas in {}: {}'.format(shap_dir, len(metas)))
for name in metas:
    with open(os.path.join(shap_dir, name), encoding='utf-8') as fh:
        d = json.load(fh)
    phi = [str(c) for c in (d.get('feature_names') or [])]
    agree = ' '.join('{}={}'.format(lab.split('.')[-1][:14], NAMES[lab] == phi)
                     for lab in NAMES)
    print('  {:<44} n={:<4} feature_order={!r:<30} {}'.format(
        name, len(phi), d.get('feature_order'), agree))
